In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
df = pd.read_csv('uber.csv').dropna()
df = df.drop(columns=[c for c in ['Unnamed: 0','key'] if c in df])
df['pickup_datetime'] = pd.to_datetime(df.pickup_datetime, errors='coerce')
df['hour'] = df.pickup_datetime.dt.hour; df['day'] = df.pickup_datetime.dt.dayofweek
df['dist'] = np.hypot(df.pickup_longitude-df.dropoff_longitude, df.pickup_latitude-df.dropoff_latitude)*111  # ~km
df = df.drop(columns='pickup_datetime').dropna()

In [ ]:
# Outliers: IQR rule
for c in ['fare_amount','dist']:
    q1, q3 = df[c].quantile([.25,.75]); i = q3-q1
    df = df[df[c].between(q1-1.5*i, q3+1.5*i)]
df = df[(df.fare_amount>0) & (df.passenger_count.between(1,6))]
df.boxplot(column=['fare_amount','dist'])

In [ ]:
df.corr()['fare_amount']

In [ ]:
X, y = df.drop(columns='fare_amount'), df.fare_amount
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=0)
for m in [LinearRegression(), RandomForestRegressor(n_estimators=50, n_jobs=-1)]:
    p = m.fit(Xtr, ytr).predict(Xte)
    print(type(m).__name__, 'R2=%.3f RMSE=%.3f' % (r2_score(yte,p), mean_squared_error(yte,p)**.5))